# Раздел 4 · Механика векторного поиска: query_points, top-K, hnsw_ef

> Этот ноутбук продолжает Раздел 3 («Коллекции и точки») — использует уже созданную там коллекцию `BOOKS_COLLECTION`. Если ниже вылетит `AssertionError: Коллекция ... ещё не создана` — сначала откройте и выполните ноутбук Раздела 3, потом возвращайтесь сюда.

In [ ]:
%pip install qdrant-client sentence-transformers ipykernel --quiet


In [ ]:
import json
import subprocess
import numpy as np

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    SearchParams, RecommendQuery, RecommendInput
)

print("OK: библиотеки импортированы")


### Docker: тот же сервер, что и в остальных ноутбуках курса

In [ ]:
CONTAINER_NAME = "qdrant_lecture"


def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip(), result.stderr.strip(), result.returncode


sh("docker volume create qdrant_storage")

_, _, exists = sh("docker inspect {name}".format(name=CONTAINER_NAME))
if exists != 0:
    out, err, _ = sh(
        "docker run -d --name {name} -p 6333:6333 -p 6334:6334 "
        "-v qdrant_storage:/qdrant/storage qdrant/qdrant".format(name=CONTAINER_NAME)
    )
else:
    out, err, _ = sh("docker start {name}".format(name=CONTAINER_NAME))
print(out or err)

client = QdrantClient(url="http://localhost:6333")
print(client.get_collections())


In [ ]:
model = SentenceTransformer("intfloat/multilingual-e5-small")
EMBED_DIM = model.get_embedding_dimension()


def embed_passages(texts):
    return model.encode(["passage: " + t for t in texts], normalize_embeddings=True).tolist()


def embed_query(text):
    return model.encode("query: " + text, normalize_embeddings=True).tolist()


def cos(x, y):
    return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))


with open("../books_dataset.json", encoding="utf-8") as f:
    BOOKS = json.load(f)
BOOK_BY_ID = {b["id"]: b for b in BOOKS}
PREFIX = "exercise1_"
BOOKS_COLLECTION = PREFIX + "books"

print("Размерность эмбеддинга:", EMBED_DIM, "| книг в подборке:", len(BOOKS))


In [ ]:
assert client.collection_exists(BOOKS_COLLECTION), (
    "Коллекция {} ещё не создана. Откройте и выполните ноутбук "
    "Раздела 3 (\"Коллекции и точки\") первым - там она создаётся и наполняется."
).format(BOOKS_COLLECTION)
print("OK:", BOOKS_COLLECTION, "-", client.count(BOOKS_COLLECTION).count, "точек")


## Механика поиска

### Пример — семантический поиск

In [ ]:
# query_points - основной метод поиска; query здесь - вектор запроса (уже с префиксом "query: " из embed_query)
example_hits = client.query_points(
    collection_name=BOOKS_COLLECTION,
    query=embed_query("короткий юмористический рассказ о чинопочитании и подхалимстве"),
    limit=5,
).points

for hit in example_hits:
    print(hit.id, round(hit.score, 3), hit.payload["genre"], "-", hit.payload["title"])


Реальный топ-1: `id=43`, "Толстый и тонкий" (Чехов), score `0.859` — ровно тот рассказ, где старый
товарищ мгновенно меняет тон, узнав чин собеседника.

## Задание 1

Найдите топ-5 по запросу **"мрачная история об убийстве и мучительных муках совести"**. Сохраните в `hits_ex1`.

**Используйте:** `client.query_points(...)`, `embed_query(...)`.

In [ ]:
# ВАШ КОД ЗДЕСЬ
hits_ex1 = ...


In [ ]:
def check_ex1(hits_ex1):
    assert len(hits_ex1) == 5, "ожидали top-5"
    assert hits_ex1[0].id == 10, (
        "первым результатом на запрос про убийство и муки совести ожидали id=10 "
        "(Преступление и наказание), получили id={}".format(hits_ex1[0].id)
    )
    print("OK: топ-1 —", hits_ex1[0].payload["title"], "score =", round(hits_ex1[0].score, 3))
    print("ОТВЕТ ДЛЯ STEPIK:", hits_ex1[0].id)


check_ex1(hits_ex1)


### Пример — recall: exact vs HNSW, один запрос

In [ ]:
qv = embed_query("трагическая история о безответной любви")

# exact=True - честный полный перебор без графа HNSW, эталон для сравнения
exact = client.query_points(BOOKS_COLLECTION, query=qv, limit=5, search_params=SearchParams(exact=True)).points
# hnsw_ef - ширина луча поиска по графу HNSW: больше = точнее, но медленнее
approx = client.query_points(BOOKS_COLLECTION, query=qv, limit=5, search_params=SearchParams(hnsw_ef=64)).points

exact_ids = {p.id for p in exact}
approx_ids = {p.id for p in approx}
print("exact ids: ", exact_ids)
print("approx ids:", approx_ids)
print("recall@5 для этого запроса:", len(exact_ids & approx_ids) / 5)


## Задание 2

Повторите то же сравнение (exact vs `hnsw_ef`) для запроса **"лёгкий сатирический рассказ о чиновниках"**
и `hnsw_ef=16` (вместо 64). Сохраните id топ-5 в `exact_ids_ex2` и `approx_ids_ex2`.

**Используйте:** `client.query_points(...)`, `SearchParams(exact=True)`, `SearchParams(hnsw_ef=16)`.

In [ ]:
# ВАШ КОД ЗДЕСЬ
exact_ids_ex2 = ...
approx_ids_ex2 = ...


In [ ]:
def check_ex2(exact_ids_ex2, approx_ids_ex2):
    assert len(exact_ids_ex2) == 5 and len(approx_ids_ex2) == 5, "ожидали top-5 в обоих случаях"
    assert set(exact_ids_ex2) == set(approx_ids_ex2), (
        "на 87 точках exact и hnsw_ef=16 должны совпасть: {} vs {}".format(exact_ids_ex2, approx_ids_ex2)
    )
    print("OK: exact и hnsw_ef=16 совпали:", exact_ids_ex2)
    print("ОТВЕТ ДЛЯ STEPIK:", exact_ids_ex2[0])


check_ex2(exact_ids_ex2, approx_ids_ex2)


### Пример — group_by

In [ ]:
# group_by схлопывает несколько хитов одного значения поля в одну группу - удобно для "не больше N на жанр/автора"
example_groups = client.query_points_groups(
    collection_name=BOOKS_COLLECTION,
    query=embed_query("история о неудачной любви и расставании"),
    group_by="genre",
    group_size=2,   # не больше 2 хитов внутри одной группы
    limit=5,        # не больше 5 групп
)

for group in example_groups.groups:
    print(group.id, "->", [h.payload["title"] for h in group.hits])


## Задание 3

То же самое, но для запроса **"история о разочаровании в жизни и внутренней пустоте"** — сохраните в `groups_ex3`.

**Используйте:** `client.query_points_groups(...)`, `embed_query(...)`.

In [ ]:
# ВАШ КОД ЗДЕСЬ
groups_ex3 = ...


In [ ]:
def check_ex3(groups_ex3):
    assert len(groups_ex3.groups) <= 5, "ожидали не больше 5 групп (limit=5)"
    for group in groups_ex3.groups:
        assert len(group.hits) <= 2, "group_size=2, а в группе {} оказалось {} хитов".format(group.id, len(group.hits))
    print("OK:", [(g.id, len(g.hits)) for g in groups_ex3.groups])
    print("ОТВЕТ ДЛЯ STEPIK:", len(groups_ex3.groups))


check_ex3(groups_ex3)


### Пример — Recommend API

`positive` — id трёх тяжёлых, тёмных романов Достоевского, `negative` — id лёгкого юмористического рассказа Чехова. Признак успеха здесь не "результат — обязательно Достоевский" (Recommend усредняет смысл, а не автора), а то, что среди результатов не окажется лёгких юмористических вещей из того же "негативного" кластера:

In [ ]:
LIGHT_COMEDY_IDS = {42, 43, 48, 49, 50}  # короткие сатирические рассказы Чехова

# RecommendInput усредняет векторы positive и отталкивается от векторов negative - это не "похоже на A, но не на B"
# буквально, а векторная арифметика поверх эмбеддингов
example_recs = client.query_points(
    collection_name=BOOKS_COLLECTION,
    query=RecommendQuery(
        recommend=RecommendInput(positive=[10, 12, 13], negative=[42]),
    ),
    limit=5,
).points

for h in example_recs:
    print(h.id, h.payload["genre"], "-", h.payload["title"])

result_ids = {h.id for h in example_recs}
print("пересечение с лёгкой комедией:", result_ids & LIGHT_COMEDY_IDS)


Реальный результат: `[6, 17, 19, 28, 14]` — Отцы и дети, Воскресение, Что делать?, Дубровский,
Униженные и оскорблённые. Ни одного результата из набора лёгких сатирических рассказов Чехова —
направление `positive`/`negative` честно увело поиск от лёгкого тона, хотя и не свело всё строго
к одному автору.

## Задание 4

Разверните направление: `positive` = id трёх лёгких сатирических рассказов Чехова (`42`, `43`, `48`),
`negative` = id тяжёлых романов Достоевского (`{10, 11, 12, 13}`). Как и в примере выше, признак успеха —
отсутствие в топ-5 книг из «негативного» кластера: здесь это любой из четырёх романов Достоевского
в датасете, а не только сам id `10`. Сохраните топ-5 в `hits_ex4`.

**Используйте:** `client.query_points(...)`, `RecommendQuery(...)`, `RecommendInput(positive=..., negative=...)`.

In [ ]:
# ВАШ КОД ЗДЕСЬ
hits_ex4 = ...


In [ ]:
HEAVY_TRAGEDY_IDS = {10, 11, 12, 13}  # все тяжёлые романы Достоевского в датасете


def check_ex4(hits_ex4):
    assert len(hits_ex4) == 5
    result_ids = {h.id for h in hits_ex4}
    overlap = result_ids & HEAVY_TRAGEDY_IDS
    assert not overlap, "среди результатов не должно быть тяжёлых трагических романов, нашли {}".format(overlap)
    print("OK:", [(h.id, h.payload["genre"], h.payload["title"]) for h in hits_ex4])
    print("ОТВЕТ ДЛЯ STEPIK:", hits_ex4[0].id)


check_ex4(hits_ex4)
